# Exercise

`단어를 나누는 다른 방법, 서브워드`의 실습 노트북입니다. [[YOUR CODE]]로 표시된 부분을 채워 실습 페이지를 완성해봅시다.

먼저 실습에 사용할 txt 파일을 다운로드하겠습니다.

In [1]:
import requests

url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
response = requests.get(url)

with open("corpus.txt", "wb") as f:
    f.write(response.content)

In [2]:
import os

file_path = "/content/corpus.txt"

if os.path.exists(file_path):
    print("다운로드 완료")
else:
    print("파일을 찾을 수 없음")

다운로드 완료


## Q1. Unigram과 BPE 방식 중 어떤 것이 더 적절할지 고르고 그 이유를 서술하십시오.

데이터가 한글로된 영화 리뷰이니 형태가 다양할 것이고
유니그램은 서브유닛들을 추정해서 확률로 계산하여 높은 확률들의 조합으로 어휘를 구성하기에 더 적합하다고 볼 수 있다고 생각합니다.

<details>
<summary> 답변 예시 </summary>

한국어에 포함된 다양한 분절 패턴을 모델이 학습 가능하고 조사와 어미의 변형을 덜 깨뜨리면서도 압축된 토큰 집합을 만들 수 있어 Unigram이 더 적절합니다.

</details>

## Q2. `SentencePiece`를 사용하여 단어사전을 학습시키십시오.

In [3]:
import pandas as pd
import sentencepiece as spm

# corpus.txt는 id, document, label로 구성된 TSV 파일
df = pd.read_csv("corpus.txt", delimiter="\t")

# 결측치 제거
df = df.dropna(subset=["document"])

# SentencePiece 학습용 txt 파일 생성
with open("spm_input.txt", "w", encoding="utf-8") as f:
    for sentence in df["document"]:
        f.write(str(sentence).strip() + "\n")

# SentencePiece 단어사전 학습
spm.SentencePieceTrainer.train(
    input="spm_input.txt",
    model_prefix="nsmc_unigram",
    vocab_size=8000,
    model_type="unigram",
    character_coverage=0.9995,
    unk_id=0,
    bos_id=1,
    eos_id=2,
    pad_id=3
)

print("SentencePiece 학습 완료")

SentencePiece 학습 완료


## Q3. `.vocab` 파일을 읽어 상위 10개 서브워드를 출력하십시오.

In [4]:
vocab_file = "nsmc_unigram.vocab"

top_subwords = []

with open(vocab_file, "r", encoding="utf-8") as f:
    for line in f:
        token, score = line.strip().split("\t")

        if token in ["<unk>", "<s>", "</s>", "<pad>"]:
            continue

        top_subwords.append((token, score))

        if len(top_subwords) == 10:
            break

for i, (token, score) in enumerate(top_subwords, start=1):
    print(f"{i}. {token}\t{score}")

1. ▁	-3.12099
2. .	-3.48609
3. 이	-4.33498
4. ..	-4.39355
5. ▁영화	-4.58017
6. ...	-4.63689
7. 가	-4.65734
8. 의	-4.66818
9. 도	-4.72959
10. 는	-4.78741


## Q4. 예시 문장을 인코딩 후 디코딩하십시오.

In [5]:
import sentencepiece as spm

sp = spm.SentencePieceProcessor()
sp.load("nsmc_unigram.model")

True

In [6]:
text = "이 영화는 정말 재미있고 배우들의 연기도 좋았다."
sample = "아 영화재미있다."

pieces = sp.encode(text, out_type=str)
ids = sp.encode(text, out_type=int)

print("원문:", text)
print("서브워드 토큰:", pieces)
print("정수 ID:", ids)

원문: 이 영화는 정말 재미있고 배우들의 연기도 좋았다.
서브워드 토큰: ['▁이', '▁영화는', '▁정말', '▁재미있고', '▁배우들의', '▁연기도', '▁좋았다', '.']
정수 ID: [29, 126, 28, 1990, 796, 556, 667, 5]


In [7]:
decoded = sp.decode(ids)

print("디코딩 결과:", decoded)


디코딩 결과: 이 영화는 정말 재미있고 배우들의 연기도 좋았다.
